In [0]:
customers_df = spark.read.table(
    "ecommerce_catalog.silver.customers"
)

products_df = spark.read.table(
    "ecommerce_catalog.silver.products"
)

orders_df = spark.read.table(
    "ecommerce_catalog.silver.orders"
)


print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Orders:", orders_df.count())

display(products_df)

In [0]:
orders_customers_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

sales_detail = orders_customers_df.join(
    products_df,
    on="product_id",
    how="inner"
)

display(sales_detail)

display(orders_customers_df)

In [0]:
from pyspark.sql.functions import col

sales_detail = sales_detail.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

sales_detail = sales_detail.select(
    "order_id",
    "order_date",
    "customer_id",
    "customer_name",
    "product_name",
    "category",
    "quantity",
    "sales_amount"
)

display(sales_detail)

In [0]:
sales_detail.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_catalog.gold.sales_detail")

In [0]:
from pyspark.sql.functions import sum, count

daily_sales_summary = (
    sales_detail
    .groupBy("order_date")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("sales_amount").alias("total_sales")
    )
    .orderBy("order_date")
)

display(daily_sales_summary)

In [0]:
daily_sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_catalog.gold.daily_sales_summary"
    )

In [0]:
%sql
select * from ecommerce_catalog.gold.daily_sales_summary

In [0]:
print("Gold sales_detail rows:", sales_detail.count())

from pyspark.sql.functions import col

sales_detail.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
sales_detail.filter(
    col("order_id").isNull() |
    col("order_date").isNull() |
    col("customer_id").isNull() |
    col("customer_name").isNull() |
    col("product_name").isNull() |
    col("category").isNull() |
    col("quantity").isNull() |
    col("sales_amount").isNull()
).show()

In [0]:
from pyspark.sql.functions import col

silver_orders = spark.read.table(
    "ecommerce_catalog.silver.orders"
)

silver_products = spark.read.table(
    "ecommerce_catalog.silver.products"
)

sales_validation = (
    silver_orders
    .join(
        silver_products.select("product_id", "price"),
        on="product_id",
        how="inner"
    )
    .withColumn(
        "expected_sales_amount",
        col("quantity") * col("price")
    )
)

display(sales_validation)

In [0]:
gold_sales = spark.read.table(
    "ecommerce_catalog.gold.sales_detail"
)

comparison = (
    gold_sales
    .join(
        sales_validation.select(
            "order_id",
            "expected_sales_amount"
        ),
        on="order_id",
        how="inner"
    )
)

display(comparison)

In [0]:
comparison.filter(
    col("sales_amount") != col("expected_sales_amount")
).show()

In [0]:
silver_customers = spark.read.table(
    "ecommerce_catalog.silver.customers"
)

invalid_customers = silver_orders.join(
    silver_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Orders with invalid customers:", invalid_customers.count())